# Stockage des données d'entraînement — MinIO + MLflow

Ce notebook illustre la **stratégie hybride** recommandée pour stocker les données d'entraînement :

| Couche | Rôle |
|--------|------|
| **MinIO** `datasets/` | Stockage physique des fichiers (parquet) — pérenne, sans duplication |
| **MLflow** `log_input` | Traçabilité — référence vers MinIO sans copier les données |
| **`model_metadata.training_dataset`** | URI consultable rapidement via l'API |

**Pourquoi pas `mlflow.log_artifact()` pour les données ?**  
Chaque run copierait le dataset dans `s3://mlflow/{run_id}/artifacts/` → duplication massive pour N runs sur le même dataset.

**Prérequis** : `docker-compose up -d`

## 1. Configuration

In [ ]:
import os
import json
from io import BytesIO

import numpy as np
import pandas as pd
import requests
from minio import Minio
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import mlflow
import mlflow.sklearn

# ── MinIO ─────────────────────────────────────────────────────────────────────
MINIO_ENDPOINT   = 'localhost:9000'
MINIO_ACCESS_KEY = 'minioadmin'
MINIO_SECRET_KEY = 'minioadmin'
DATASETS_BUCKET  = 'datasets'

# ── MLflow ────────────────────────────────────────────────────────────────────
MLFLOW_TRACKING_URI = 'http://localhost:5000'

os.environ['MLFLOW_S3_ENDPOINT_URL'] = f'http://{MINIO_ENDPOINT}'
os.environ['AWS_ACCESS_KEY_ID']      = MINIO_ACCESS_KEY
os.environ['AWS_SECRET_ACCESS_KEY']  = MINIO_SECRET_KEY

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# ── API predictml ─────────────────────────────────────────────────────────────
API_URL   = 'http://localhost:8000'
API_TOKEN = 'ZC_W_-mcw-01l5W5fN8VFx-h4WornlnxwAtiQutT2BA'
HEADERS   = {'Authorization': f'Bearer {API_TOKEN}'}

# ── Modèle ────────────────────────────────────────────────────────────────────
MODEL_NAME    = 'iris_rf_model'
MODEL_VERSION = '1.0.0'
DATASET_NAME  = 'iris_synthetic'
DATASET_VERSION = 'v1'
DATASET_URI   = f's3://{DATASETS_BUCKET}/{DATASET_NAME}/{DATASET_VERSION}'

print(f'MinIO        : {MINIO_ENDPOINT}')
print(f'MLflow       : {MLFLOW_TRACKING_URI}')
print(f'Dataset URI  : {DATASET_URI}')
print(f'Modèle       : {MODEL_NAME} v{MODEL_VERSION}')

## 2. Génération du dataset

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
df = iris.frame.copy()
df['species'] = df['target'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

FEATURE_COLS = list(iris.feature_names)
TARGET       = 'target'

X = df[FEATURE_COLS]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

df_train = X_train.copy(); df_train['target'] = y_train
df_test  = X_test.copy();  df_test['target']  = y_test

print(f'Train : {df_train.shape}  |  Test : {df_test.shape}')
df.head()

## 3. Upload du dataset dans MinIO (`datasets/`)

Les données sont stockées dans un bucket **séparé** du bucket `mlflow/` utilisé pour les artifacts de run.
Elles sont ainsi pérennes, indépendantes du cycle de vie des expériences MLflow.

In [ ]:
minio_client = Minio(
    MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=False,
)

# Créer le bucket s'il n'existe pas
if not minio_client.bucket_exists(DATASETS_BUCKET):
    minio_client.make_bucket(DATASETS_BUCKET)
    print(f"Bucket '{DATASETS_BUCKET}' créé.")
else:
    print(f"Bucket '{DATASETS_BUCKET}' existe déjà.")

def upload_parquet(df: pd.DataFrame, object_path: str) -> int:
    """Sérialise un DataFrame en parquet et l'uploade dans MinIO."""
    buf = BytesIO()
    df.to_parquet(buf, index=False)
    buf.seek(0)
    size = buf.getbuffer().nbytes
    minio_client.put_object(
        DATASETS_BUCKET, object_path,
        buf, length=size,
        content_type='application/octet-stream',
    )
    return size

train_path = f'{DATASET_NAME}/{DATASET_VERSION}/train.parquet'
test_path  = f'{DATASET_NAME}/{DATASET_VERSION}/test.parquet'

train_size = upload_parquet(df_train, train_path)
test_size  = upload_parquet(df_test,  test_path)

print(f'Uploadé : {DATASETS_BUCKET}/{train_path}  ({train_size} bytes)')
print(f'Uploadé : {DATASETS_BUCKET}/{test_path}   ({test_size} bytes)')
print(f'\nDataset URI : {DATASET_URI}')

## 4. Entraînement + Run MLflow

Le dataset est **référencé** via `mlflow.log_input()` — MLflow ne copie pas les données,
il stocke uniquement la source URI. L'URI est aussi loggée en paramètre pour un accès
rapide depuis l'UI MLflow.

In [ ]:
RF_PARAMS = {
    'n_estimators': 100,
    'max_depth':    5,
    'random_state': 42,
}

model = RandomForestClassifier(**RF_PARAMS)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
metrics = {
    'accuracy':    float(accuracy_score(y_test, y_pred)),
    'f1_weighted': float(f1_score(y_test, y_pred, average='weighted')),
}
print('Métriques :', metrics)

In [ ]:
mlflow.set_experiment(MODEL_NAME)

with mlflow.start_run(run_name=f'{MODEL_NAME}_v{MODEL_VERSION}') as run:

    # ── Référence au dataset (sans copie des données) ─────────────────────────
    # mlflow.log_input stocke uniquement la source URI dans les métadonnées du run.
    # Les données restent dans MinIO datasets/, aucune duplication.
    dataset = mlflow.data.from_numpy(
        X_train.to_numpy(),
        source=f'{DATASET_URI}/train.parquet',
        name=f'{DATASET_NAME}_{DATASET_VERSION}_train',
    )
    mlflow.log_input(dataset, context='training')

    # ── Paramètres ───────────────────────────────────────────────────────────
    mlflow.log_params({
        **{f'rf_{k}': v for k, v in RF_PARAMS.items()},
        # URI loggée en param pour visibilité directe dans l'UI MLflow
        'dataset_uri':     DATASET_URI,
        'dataset_version': DATASET_VERSION,
        'n_train':         len(X_train),
        'n_test':          len(X_test),
        'features':        ', '.join(FEATURE_COLS),
    })

    # ── Métriques ─────────────────────────────────────────────────────────────
    mlflow.log_metrics(metrics)

    # ── Tags ──────────────────────────────────────────────────────────────────
    mlflow.set_tags({
        'dataset_bucket': DATASETS_BUCKET,
        'task':           'multiclass_classification',
        'framework':      'scikit-learn',
    })

    # ── Modèle ────────────────────────────────────────────────────────────────
    mlflow.sklearn.log_model(model, artifact_path='model')

    RUN_ID = run.info.run_id
    print(f'Run ID       : {RUN_ID}')
    print(f'MLflow UI    : {MLFLOW_TRACKING_URI}/#/experiments/{run.info.experiment_id}/runs/{RUN_ID}')
    print(f'Dataset URI  : {DATASET_URI}  (référence, aucune copie)')
    print(f'Accuracy     : {metrics["accuracy"]:.4f}')

## 5. Enregistrement via l'API predictml

Le champ `training_dataset` reçoit l'URI MinIO — il est persisté dans `model_metadata.training_dataset`
et consultable via `GET /models/{name}/{version}`.

In [ ]:
payload = {
    'name':             MODEL_NAME,
    'version':          MODEL_VERSION,
    'description':      'RandomForest Iris — exemple stockage dataset MinIO + MLflow',
    'algorithm':        'RandomForestClassifier',
    'mlflow_run_id':    RUN_ID,
    'accuracy':         metrics['accuracy'],
    'f1_score':         metrics['f1_weighted'],
    'features_count':   len(FEATURE_COLS),
    'classes':          json.dumps([0, 1, 2]),
    'training_dataset': DATASET_URI,   # s3://datasets/iris_synthetic/v1
    'training_params':  json.dumps(RF_PARAMS),
}

resp = requests.post(f'{API_URL}/models', headers=HEADERS, data=payload)

if resp.status_code == 201:
    print(f'Modèle créé — id={resp.json()["id"]}')
elif resp.status_code == 409:
    print('Modèle existe déjà (409) — OK.')
else:
    print(f'Erreur {resp.status_code}: {resp.text}')

In [ ]:
# Vérifier que training_dataset est bien persisté
resp = requests.get(f'{API_URL}/models/{MODEL_NAME}/{MODEL_VERSION}', headers=HEADERS)
meta = resp.json()

print(f'Modèle          : {meta["name"]} v{meta["version"]}')
print(f'MLflow run_id   : {meta["mlflow_run_id"]}')
print(f'training_dataset: {meta["training_dataset"]}')
print(f'Accuracy        : {meta["accuracy"]}')

## 6. Relire le dataset depuis MinIO

À tout moment, on peut récupérer les données d'entraînement à partir de l'URI stockée dans les métadonnées.

In [ ]:
def download_parquet(bucket: str, object_path: str) -> pd.DataFrame:
    """Télécharge un fichier parquet depuis MinIO et retourne un DataFrame."""
    response = minio_client.get_object(bucket, object_path)
    buf = BytesIO(response.read())
    response.close()
    response.release_conn()
    return pd.read_parquet(buf)

# Récupérer l'URI depuis les métadonnées de l'API
stored_uri = meta['training_dataset']        # 's3://datasets/iris_synthetic/v1'
bucket     = stored_uri.split('/')[2]        # 'datasets'
prefix     = '/'.join(stored_uri.split('/')[3:])  # 'iris_synthetic/v1'

df_train_reloaded = download_parquet(bucket, f'{prefix}/train.parquet')
df_test_reloaded  = download_parquet(bucket, f'{prefix}/test.parquet')

print(f'Train rechargé : {df_train_reloaded.shape}')
print(f'Test rechargé  : {df_test_reloaded.shape}')
df_train_reloaded.head()

## Récapitulatif

| Étape | Stockage | Détail |
|-------|----------|--------|
| Dataset train/test | MinIO `datasets/iris_synthetic/v1/` | Parquet, une seule copie |
| Référence dataset | MLflow `log_input` | URI uniquement, pas de copie |
| URI en paramètre | MLflow `log_param("dataset_uri", ...)` | Visible dans l'UI MLflow |
| Modèle | MLflow artifacts `s3://mlflow/{run_id}/model` | Via `mlflow.sklearn.log_model` |
| Métadonnées | PostgreSQL `model_metadata.training_dataset` | URI stockée, consultable via API |

**Résultat** : 10 re-entraînements sur le même dataset = **1 seule copie** des données dans MinIO,
10 runs MLflow avec référence traçable, 0 duplication.